# Training YOLO: Deteksi Helm Pengendara Motor

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/projects/helmet-detection/notebooks/01_train_yolo_colab.ipynb)

Notebook ini melatih (*fine-tune*) model YOLO pretrained untuk mendeteksi pengendara motor yang **memakai helm** dan **tidak memakai helm**, dari data mentah sampai model siap pakai:

1. Cek GPU, clone repo, dan instal dependensi.
2. Siapkan dataset dengan `src/prepare_data.py` (unduh, susun, validasi).
3. Visualisasi contoh label.
4. Training dengan checkpoint di Google Drive, sehingga bisa dilanjutkan (*resume*) jika Colab terputus.
5. Evaluasi di split test: mAP, precision/recall per kelas, confusion matrix.
6. Contoh prediksi dan penyimpanan hasil ke `results/`.

**Sebelum mulai:**
- Aktifkan GPU: **Runtime → Change runtime type → T4 GPU**.
- Simpan URL dataset di **Colab Secrets** (ikon 🔑 di sidebar kiri) dengan nama `DATASET_URL`, lalu aktifkan *Notebook access*. Dengan begitu URL yang memuat API key tidak tertulis di notebook.

> ⚖️ `ultralytics` dan bobot YOLO berlisensi **AGPL-3.0**; repo ini juga AGPL-3.0 (lihat README proyek).

## 0. Konfigurasi

Semua pengaturan penting ada di sel ini. Ubah sesuai kebutuhan, lalu jalankan semua sel dari atas (**Runtime → Run all**).

| Parameter | Penjelasan |
|---|---|
| `MODEL` | Bobot awal. `yolo11n.pt` (cepat) untuk percobaan, `yolo11s.pt`/`yolo11m.pt` untuk akurasi lebih tinggi. |
| `EPOCHS`, `PATIENCE` | Jumlah epoch maksimum; training berhenti lebih awal jika mAP validasi tidak membaik selama `PATIENCE` epoch. |
| `IMGSZ` | Ukuran input. 640 standar; naikkan (misal 960) jika banyak helm berukuran kecil di gambar, dengan biaya lebih lambat. |
| `BATCH` | `-1` = otomatis menyesuaikan memori GPU (sekitar 60%). |
| `DATASET_SOURCE` | Kosongkan untuk memakai Secret `DATASET_URL`, atau isi path file zip (misal di Google Drive). |
| `SAVE_TO_DRIVE` | Simpan hasil training di Google Drive agar tidak hilang saat runtime Colab berakhir. |

`QUICK_RUN=1` (environment variable) dipakai untuk uji cepat di CPU: 1 epoch dan gambar kecil.

In [ ]:
import os

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"

# --- Repo ---
REPO_URL = "https://github.com/riki-profile/ai-engineer-journey.git"
REPO_BRANCH = "main"

# --- Dataset ---
DATASET_SOURCE = ""          # kosong = pakai Secret/env DATASET_URL; atau path zip, mis. "/content/drive/MyDrive/helm.zip"
STOP_ON_DATA_ERROR = True    # hentikan notebook jika validasi dataset menemukan error

# --- Training ---
MODEL = "yolo11n.pt"
EPOCHS = 1 if QUICK_RUN else 100
PATIENCE = 20
IMGSZ = 320 if QUICK_RUN else 640
BATCH = 8 if QUICK_RUN else -1
SEED = 42
RUN_NAME = f"helm_{MODEL.removesuffix('.pt')}_{IMGSZ}"

# --- Penyimpanan ---
SAVE_TO_DRIVE = not QUICK_RUN
DRIVE_DIR = "/content/drive/MyDrive/helmet-detection"

print(f"QUICK_RUN={QUICK_RUN} | model={MODEL} | epochs={EPOCHS} | imgsz={IMGSZ} | run={RUN_NAME}")

## 1. Cek GPU

Training YOLO di CPU sangat lambat (bisa berjam-jam per epoch untuk dataset ribuan gambar). Pastikan sel ini menampilkan GPU sebelum training penuh.

In [ ]:
import shutil
import subprocess
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU terdeteksi: {props.name} | memori {props.total_memory / 1e9:.1f} GB")
    DEVICE = 0
else:
    print("⚠️ GPU tidak terdeteksi, training akan sangat lambat di CPU.")
    print("   Di Colab: Runtime → Change runtime type → T4 GPU")
    DEVICE = "cpu"

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])

## 2. Clone Repo dan Instal Dependensi

Jika notebook dijalankan dari dalam repo (misalnya di komputer lokal), proses clone dilewati. Jika belum, repo di-clone ke `/content`.

Jika repo bersifat **private**, simpan GitHub *personal access token* (izin baca repo) di Colab Secrets dengan nama `GITHUB_TOKEN`. Token hanya dipakai untuk clone dan tidak dicetak.

In [ ]:
from pathlib import Path


def get_secret(nama):
    # Ambil dari Colab Secrets jika tersedia, jika tidak dari environment variable
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:
        return os.environ.get(nama)


def cari_folder_proyek():
    for kandidat in [Path.cwd(), Path.cwd().parent]:
        if (kandidat / "configs" / "data.yaml").exists() and (kandidat / "src" / "prepare_data.py").exists():
            return kandidat
    return None


PROJECT_DIR = cari_folder_proyek()
if PROJECT_DIR is None:
    tujuan = Path("/content/ai-engineer-journey")
    if not tujuan.exists():
        token = get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
        hasil = subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, str(tujuan)],
                               capture_output=True, text=True)
        if hasil.returncode != 0:
            # Sembunyikan token jika muncul di pesan error
            raise RuntimeError("Clone gagal:\n" + hasil.stderr.replace(token or "\0", "***"))
    PROJECT_DIR = tujuan / "projects" / "helmet-detection"

os.chdir(PROJECT_DIR)
print("Folder kerja:", Path.cwd())

Instal dependensi proyek (`ultralytics`, `python-dotenv`). PyTorch sudah tersedia di Colab.

In [ ]:
%pip install -q -r requirements.txt

## 3. Google Drive (Opsional)

Runtime Colab bersifat sementara: semua file hilang saat sesi berakhir atau terputus. Dengan `SAVE_TO_DRIVE = True`, folder hasil training disimpan langsung di Google Drive, sehingga checkpoint aman dan training bisa dilanjutkan.

In [ ]:
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        RUNS_DIR = Path(DRIVE_DIR) / "runs"
    except ImportError:
        print("Bukan di Colab, hasil disimpan lokal.")
        RUNS_DIR = PROJECT_DIR / "runs"
else:
    RUNS_DIR = PROJECT_DIR / "runs"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
print("Hasil training disimpan di:", RUNS_DIR)

## 4. Menyiapkan Dataset

`src/prepare_data.py` mengunduh dataset, menyusunnya ke `data/images|labels/{train,val,test}`, menyalin nama kelas ke `configs/data.yaml` (`--sync-names`), lalu memvalidasi format YOLO.

Jika folder `data/` sudah ada (misalnya sel ini dijalankan ulang), skrip hanya melakukan validasi.

URL dataset dikirim lewat environment variable, bukan argumen perintah, agar tidak tercetak di output notebook.

In [ ]:
import sys

sumber = DATASET_SOURCE or get_secret("DATASET_URL")
env = dict(os.environ)

if (PROJECT_DIR / "data" / "images").exists():
    perintah = [sys.executable, "src/prepare_data.py", "--validate-only"]
elif sumber:
    env["DATASET_URL"] = sumber
    perintah = [sys.executable, "src/prepare_data.py", "--sync-names"]
else:
    raise RuntimeError("Dataset belum diatur: isi Secret DATASET_URL atau variabel DATASET_SOURCE di sel konfigurasi.")

hasil = subprocess.run(perintah, env=env, capture_output=True, text=True)
print(hasil.stdout[-6000:])
if hasil.returncode != 0:
    print(hasil.stderr[-3000:])
    if STOP_ON_DATA_ERROR:
        raise RuntimeError("Validasi dataset gagal. Perbaiki error di atas, atau set STOP_ON_DATA_ERROR = False.")

Nama kelas yang akan dipakai model (dari `configs/data.yaml`). Pastikan urutannya sesuai dengan label dataset.

In [ ]:
import json
import yaml

DATA_YAML = PROJECT_DIR / "configs" / "data.yaml"
data_cfg = yaml.safe_load(DATA_YAML.read_text())
CLASS_NAMES = {int(k): v for k, v in data_cfg["names"].items()}
laporan = json.loads((PROJECT_DIR / "results" / "data_report.json").read_text())
ADA_TEST = laporan["split"]["test"]["gambar"] > 0

print("Kelas:", CLASS_NAMES)
for split, s in laporan["split"].items():
    print(f"  {split:<5}: {s['gambar']:>5} gambar | objek per kelas: {s['objek_per_kelas']}")

## 5. Visualisasi Contoh Label

Selalu cek label secara visual sebelum training. Kotak yang bergeser atau kelas yang tertukar akan langsung terlihat di sini.

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

WARNA = ["tab:green", "tab:red", "tab:blue", "tab:orange"]


def tampilkan_label(path_img, ax):
    img = Image.open(path_img).convert("RGB")
    w, h = img.size
    ax.imshow(img)
    path_lbl = PROJECT_DIR / "data" / "labels" / path_img.parent.name / (path_img.stem + ".txt")
    if path_lbl.exists():
        for baris in path_lbl.read_text().split("\n"):
            if not baris.strip():
                continue
            k, cx, cy, bw, bh = baris.split()[:5]
            k, cx, cy, bw, bh = int(k), float(cx) * w, float(cy) * h, float(bw) * w, float(bh) * h
            ax.add_patch(patches.Rectangle((cx - bw / 2, cy - bh / 2), bw, bh, fill=False,
                                           edgecolor=WARNA[k % len(WARNA)], lw=2))
            ax.text(cx - bw / 2, cy - bh / 2, CLASS_NAMES.get(k, k), color="white", fontsize=7,
                    va="bottom", bbox=dict(facecolor=WARNA[k % len(WARNA)], pad=1, edgecolor="none"))
    ax.axis("off")


train_imgs = sorted((PROJECT_DIR / "data" / "images" / "train").glob("*"))
random.Random(SEED).shuffle(train_imgs)
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
for ax, p in zip(axes.flat, train_imgs[:8]):
    tampilkan_label(p, ax)
plt.tight_layout()
plt.show()

## 6. Training

Kita *fine-tune* model YOLO yang sudah dilatih di COCO. COCO sudah punya kelas `person` dan `motorcycle`, jadi fitur yang dipelajari model sangat relevan untuk kasus ini.

Hal penting:
- **Resume otomatis:** jika Colab terputus di tengah training, jalankan ulang notebook dengan `RUN_NAME` yang sama; training dilanjutkan dari `last.pt`, bukan diulang dari awal. Jika run dengan nama itu sudah **selesai**, training dilewati dan bobot yang ada langsung dipakai untuk evaluasi.
- `patience`: *early stopping*, yaitu berhenti jika mAP validasi tidak membaik.
- Augmentasi bawaan ultralytics (mosaic, flip, HSV, dan sebagainya) aktif secara default. `fliplr=0.5` aman untuk kasus ini karena helm tetap helm walaupun gambar dibalik horizontal.
- Hasil (bobot, grafik, `results.csv`) ada di `RUNS_DIR/RUN_NAME/`.

In [ ]:
from ultralytics import YOLO

run_dir = RUNS_DIR / RUN_NAME
last_ckpt = run_dir / "weights" / "last.pt"


def status_checkpoint(path):
    # 'baru'    : belum ada checkpoint
    # 'terputus': training berhenti di tengah jalan -> bisa di-resume
    # 'selesai' : training sudah tuntas (ultralytics menandai epoch = -1 dan membuang state optimizer)
    if not path.exists():
        return "baru"
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    return "terputus" if ckpt.get("epoch", -1) >= 0 and ckpt.get("optimizer") is not None else "selesai"


status = status_checkpoint(last_ckpt)
if status == "selesai":
    # Jangan panggil resume di sini: pada run yang sudah selesai, ultralytics justru memulai
    # training baru dengan dataset contoh bawaan, bukan dataset kita.
    print(f"Run '{RUN_NAME}' sudah selesai sebelumnya, training dilewati dan bobot yang ada dipakai.")
    print("Untuk training ulang, ganti RUN_NAME di sel konfigurasi atau hapus folder", run_dir)
elif status == "terputus":
    print(f"Checkpoint terputus ditemukan, melanjutkan training dari {last_ckpt}")
    model = YOLO(str(last_ckpt))
    model.train(resume=True)
else:
    model = YOLO(MODEL)
    model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        patience=PATIENCE,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        seed=SEED,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,       # tulis ke folder yang sama agar resume bisa menemukannya
        workers=2,
        plots=True,
    )

BEST = run_dir / "weights" / "best.pt"
print("Bobot terbaik:", BEST)

## 7. Kurva Training

`results.csv` mencatat loss dan metrik di setiap epoch:
- **box_loss**: seberapa tepat posisi kotak.
- **cls_loss**: seberapa tepat klasifikasi helm / tanpa helm.
- **dfl_loss**: kualitas batas kotak.
- **mAP50** dan **mAP50-95** di data validasi.

Jika loss train terus turun tetapi mAP validasi stagnan atau turun, model mulai overfitting.

In [ ]:
import pandas as pd

df = pd.read_csv(run_dir / "results.csv")
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for k in ["box_loss", "cls_loss", "dfl_loss"]:
    axes[0].plot(df["epoch"], df[f"train/{k}"], "o-", label=f"train {k}")
    axes[1].plot(df["epoch"], df[f"val/{k}"], "o-", label=f"val {k}")
axes[2].plot(df["epoch"], df["metrics/mAP50(B)"], "o-", label="mAP50")
axes[2].plot(df["epoch"], df["metrics/mAP50-95(B)"], "o-", label="mAP50-95")
for ax, judul in zip(axes, ["Loss train", "Loss validasi", "mAP validasi"]):
    ax.set_title(judul); ax.set_xlabel("epoch"); ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(df.tail(3).to_string(index=False))

## 8. Evaluasi di Split Test

Data **test** tidak pernah dipakai selama training maupun pemilihan model, jadi angka di sini adalah estimasi performa yang jujur. Jika dataset tidak punya split test, evaluasi memakai val (dan hasilnya cenderung sedikit optimistis).

Metrik per kelas penting untuk kasus ini: model bisa sangat bagus mendeteksi `helmet` tetapi buruk di `no_helmet`, padahal justru kelas `no_helmet` yang paling penting untuk pemantauan kepatuhan.

In [ ]:
best_model = YOLO(str(BEST))
EVAL_SPLIT = "test" if ADA_TEST else "val"
metrics = best_model.val(data=str(DATA_YAML), split=EVAL_SPLIT, imgsz=IMGSZ, device=DEVICE,
                         project=str(RUNS_DIR), name=f"{RUN_NAME}_eval_{EVAL_SPLIT}", exist_ok=True,
                         plots=True, verbose=False)

print(f"Evaluasi di split: {EVAL_SPLIT}")
print(f"  precision : {metrics.box.mp:.3f}")
print(f"  recall    : {metrics.box.mr:.3f}")
print(f"  mAP50     : {metrics.box.map50:.3f}")
print(f"  mAP50-95  : {metrics.box.map:.3f}")

per_kelas = pd.DataFrame([
    dict(zip(["kelas", "precision", "recall", "AP50", "AP50-95"],
             [metrics.names[k], *metrics.box.class_result(i)]))
    for i, k in enumerate(metrics.box.ap_class_index)
])
print()
print(per_kelas.round(3).to_string(index=False))

### 8.1 Confusion matrix

Baris = prediksi, kolom = label asli (konvensi ultralytics). Kolom/baris **background** menunjukkan objek yang tidak terdeteksi (FN) dan deteksi palsu (FP). Hal yang perlu diperhatikan: seberapa sering `no_helmet` tertukar dengan `helmet`. Kesalahan ini paling merugikan untuk kasus pemantauan.

In [ ]:
eval_dir = Path(metrics.save_dir)
cm_file = eval_dir / "confusion_matrix_normalized.png"
if cm_file.exists():
    plt.figure(figsize=(7, 6))
    plt.imshow(Image.open(cm_file))
    plt.axis("off")
    plt.show()
else:
    # Fallback: tampilkan matriks mentah jika file gambar tidak dibuat
    cm = metrics.confusion_matrix.matrix
    label = [metrics.names[i] for i in range(len(metrics.names))] + ["background"]
    print(pd.DataFrame(cm.astype(int), index=label, columns=label))

## 9. Contoh Prediksi

Prediksi di beberapa gambar split evaluasi. Gunakan sel ini untuk mencari **pola kegagalan**: objek kecil/jauh, oklusi, malam hari, topi atau tudung yang terlihat seperti helm, dan sebagainya. Catat temuannya di README proyek.

In [ ]:
eval_imgs = sorted((PROJECT_DIR / "data" / "images" / EVAL_SPLIT).glob("*"))
random.Random(SEED).shuffle(eval_imgs)
contoh = eval_imgs[:8]
preds = best_model.predict(contoh, imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, r in zip(axes.flat, preds):
    ax.imshow(r.plot()[..., ::-1])  # BGR -> RGB
    jumlah = {n: int((r.boxes.cls == k).sum()) for k, n in CLASS_NAMES.items()}
    ax.set_title(", ".join(f"{n}: {j}" for n, j in jumlah.items()), fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 10. Menyimpan Hasil ke `results/`

File kecil yang layak di-commit (metrik, tabel per kelas, grafik) disalin ke `results/<RUN_NAME>/`. **Bobot model (`best.pt`) tidak di-commit** (sudah diabaikan `.gitignore`); bobot tetap tersimpan di `RUNS_DIR` (Google Drive jika aktif).

Di Colab, sel ini juga membuat file zip `results/` yang bisa diunduh, lalu di-commit dari komputer lokal.

In [ ]:
out_dir = PROJECT_DIR / "results" / RUN_NAME
out_dir.mkdir(parents=True, exist_ok=True)

ringkasan = {
    "model_awal": MODEL,
    "epochs_diminta": EPOCHS,
    "epochs_selesai": int(df["epoch"].max()),
    "imgsz": IMGSZ,
    "split_evaluasi": EVAL_SPLIT,
    "kelas": CLASS_NAMES,
    "precision": round(float(metrics.box.mp), 4),
    "recall": round(float(metrics.box.mr), 4),
    "mAP50": round(float(metrics.box.map50), 4),
    "mAP50-95": round(float(metrics.box.map), 4),
    "per_kelas": per_kelas.round(4).to_dict(orient="records"),
}
(out_dir / "metrics.json").write_text(json.dumps(ringkasan, indent=2, ensure_ascii=False))
per_kelas.round(4).to_csv(out_dir / "per_class.csv", index=False)
shutil.copy2(run_dir / "results.csv", out_dir / "training_log.csv")
for f in ["results.png", "confusion_matrix_normalized.png", "BoxPR_curve.png", "PR_curve.png"]:
    for folder in [eval_dir, run_dir]:
        if (folder / f).exists():
            shutil.copy2(folder / f, out_dir / f)
            break

print("Disimpan di", out_dir)
for f in sorted(out_dir.iterdir()):
    print(f"  {f.name:<36} {f.stat().st_size / 1024:>7.1f} KB")

try:
    from google.colab import files
    arsip = shutil.make_archive(f"/content/results_{RUN_NAME}", "zip", PROJECT_DIR / "results")
    files.download(arsip)
except ImportError:
    pass

## Langkah Selanjutnya

- **Catat hasil** di README proyek: model, epoch, mAP50/mAP50-95 per kelas, dan temuan dari contoh prediksi.
- **Eksperimen**: bandingkan `yolo11n` vs `yolo11s`, `IMGSZ` 640 vs 960 (helm yang jauh berukuran kecil), atau ubah `RUN_NAME` agar setiap eksperimen tersimpan terpisah.
- **Ketidakseimbangan kelas**: jika `no_helmet` jauh lebih sedikit, tambah data kelas tersebut atau periksa recall-nya secara khusus.
- **Ambang confidence**: pilih `conf` berdasarkan kurva PR. Untuk pemantauan, recall `no_helmet` yang tinggi biasanya lebih penting daripada precision.
- **Deployment**: ekspor model (`best_model.export(format="onnx")`) untuk demo. Ingat kewajiban AGPL-3.0 jika demo dibuka untuk publik.